# **Data Mining**
#### DBSCAN Practice

## Purpose

In this practice, you will use **DBSCAN** to detect unusual credit card transactions. The method is **unsupervised**: the algorithm will not use the fraud label during training. The label `Class` will be used only at the end to evaluate whether the points marked as noise are related to actual fraud cases.

## Learning goals

By the end of the activity, you should be able to:

- Explain how DBSCAN uses density to form clusters and detect noise.
- Prepare a numerical dataset for a distance-based algorithm.
- Select tentative DBSCAN parameters using a k-distance plot.
- Treat DBSCAN noise points as possible anomalies.
- Evaluate anomaly flags against known labels using precision, recall, F1-score, and a confusion matrix.

## DBSCAN in one page

**DBSCAN** stands for *Density-Based Spatial Clustering of Applications with Noise*. It groups observations that are close to many other observations and marks isolated observations as noise. In anomaly detection, those noise points are treated as candidates for unusual behavior.

### Main parameters

- **eps**: neighborhood radius. Two points are neighbors if their distance is at most `eps`.
- **min_samples**: minimum number of observations required to form a dense region.
- **Metric**: distance function used by DBSCAN. In this activity, use Euclidean distance after scaling.

### Relationship schema

```
Scaled transaction vector
        │
        ▼
  eps-neighborhood (distance radius)
        │
        ▼
   neighbors ≥ min_samples?
    ┌───┴───┐
   yes      no
    │        │
    ▼        ▼
Core point   within eps of a core point?
(starts or    ┌───┴───┐
 expands      yes     no
 cluster)      │       │
    │          ▼       ▼
    │    Border point  Noise point (label -1)
    │    (belongs to   → Possible anomaly /
    │     a cluster)     candidate fraud alert
    └──────────┘
```

### Important interpretation rule

> A DBSCAN noise point is **not** automatically a fraudulent transaction. It only means that the transaction is located in a low-density region of the feature space. The business interpretation is: this transaction deserves attention because it does not look similar to most transactions in the sample.


## 1. Setup and dataset access — 10 pts

### Dataset

Use the **Credit Card Fraud Detection** dataset from Kaggle:

https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

The file you need is `creditcard.csv`. It contains anonymized numerical features `V1` to `V28`, `Time`, `Amount`, and `Class`. The target label is `Class`, where **1** means fraud and **0** means legitimate.



In [ ]:
%pip install pandas numpy matplotlib scikit-learn kaggle


```bash
python -m venv dbscan-env
source dbscan-env/bin/activate          # macOS/Linux
# dbscan-env\Scripts\activate         # Windows PowerShell

pip install pandas numpy matplotlib scikit-learn kaggle

mkdir -p ~/.kaggle
cp kaggle.json ~/.kaggle/
chmod 600 ~/.kaggle/kaggle.json
kaggle datasets download -d mlg-ulb/creditcardfraud
unzip -o creditcardfraud.zip
```

### Grading evidence — 10 pts

- **4 pts**: The notebook imports all required libraries without errors.
- **4 pts**: The file `creditcard.csv` is loaded successfully.
- **2 pts**: The dataset shape and first rows are displayed.


## 2. Data loading and first inspection — 15 pts

Run the following cell. Keep the output visible in your notebook.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("creditcard.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


Check the expected columns and the target distribution.


In [ ]:
expected_columns = {"Time", "Amount", "Class"} | {f"V{i}" for i in range(1, 29)}
missing_columns = expected_columns - set(df.columns)

print("Missing expected columns:", missing_columns)
print("\nClass counts:")
print(df["Class"].value_counts().sort_index())
print("\nClass percentage:")
print(df["Class"].value_counts(normalize=True).sort_index().mul(100).round(4))

assert len(missing_columns) == 0, "Some expected columns are missing."
assert df["Class"].isin([0, 1]).all(), "Class must contain only 0 and 1."


Inspect missing values and basic statistics for `Time`, `Amount`, and `Class`.


In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
display(missing_summary.head(10))
display(df[["Time", "Amount", "Class"]].describe().T)
display(df.groupby("Class")["Amount"].describe())


Create two simple plots.


In [ ]:
ax = df["Class"].value_counts().sort_index().plot(kind="bar")
ax.set_title("Class distribution")
ax.set_xlabel("Class: 0 = legitimate, 1 = fraud")
ax.set_ylabel("Number of transactions")
plt.show()

ax = df["Amount"].plot(kind="hist", bins=80)
ax.set_title("Transaction amount distribution")
ax.set_xlabel("Amount")
plt.show()


### Grading evidence — 15 pts

- **5 pts**: Column validation and class distribution are shown.
- **4 pts**: Missing value check is included.
- **4 pts**: Descriptive statistics are displayed.
- **2 pts**: Two basic plots are included.


## 3. Feature preparation — 15 pts

DBSCAN is distance-based. Features with larger scales can dominate the distance calculation, so all input variables must be **scaled**. The label `Class` must **not** be included in the DBSCAN input matrix.

For classroom efficiency, use a guided subset that contains all fraud cases and a sample of legitimate transactions. This keeps the runtime manageable and gives enough fraud cases for evaluation. The label is used only to build an educational subset and later evaluate the result; it is **not** used by DBSCAN during fitting.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

RANDOM_STATE = 42
N_NORMAL = 2000

normal_df = df[df["Class"] == 0].sample(n=N_NORMAL, random_state=RANDOM_STATE)
fraud_df = df[df["Class"] == 1]
work_df = (
    pd.concat([normal_df, fraud_df], axis=0)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print("Working dataset shape:", work_df.shape)
print(work_df["Class"].value_counts().sort_index())


Build the input matrix, scale it, and create the lower-dimensional modeling matrix used by DBSCAN.


In [ ]:
feature_cols = [col for col in work_df.columns if col != "Class"]
X = work_df[feature_cols].copy()
y = work_df["Class"].astype(int).copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Input matrix shape:", X_scaled.shape)
print("Number of labels stored for evaluation:", y.shape[0])


Create a two-dimensional PCA projection **only for visualization**. DBSCAN will be fitted on a **10-dimensional** PCA modeling matrix created from the scaled features. This keeps the practice computationally manageable and reduces the effect of high-dimensional distance concentration.


In [ ]:
pca_model = PCA(n_components=10, random_state=RANDOM_STATE)
X_model = pca_model.fit_transform(X_scaled)

print("Modeling matrix shape:", X_model.shape)
print("Explained variance in 10 components:", round(pca_model.explained_variance_ratio_.sum(), 4))

pca_df = pd.DataFrame(X_model[:, :2], columns=["PC1", "PC2"])
pca_df["Class"] = y.values

ax = pca_df.plot(kind="scatter", x="PC1", y="PC2", c="Class", s=8, alpha=0.6)
ax.set_title("Two-dimensional PCA projection")
plt.show()


### Grading evidence — 15 pts

- **5 pts**: The working subset is created correctly.
- **5 pts**: `Class` is excluded from `X` and stored separately as `y`.
- **3 pts**: Standard scaling is applied to all input features.
- **2 pts**: A PCA modeling matrix and 2D visualization are included.


## 4. Choosing DBSCAN parameters — 20 pts

The two key parameters are **eps** and **min_samples**. A common starting point is to set `min_samples` between the number of dimensions and a smaller practical value. For this activity, start with `min_samples = 15` to keep the model interpretable and computationally manageable.

A **k-distance plot** helps you choose a tentative `eps`. For each point, compute the distance to its k-th nearest neighbor. Sort those distances from smallest to largest and look for an elbow or sudden increase.


In [ ]:
from sklearn.neighbors import NearestNeighbors

min_samples = 15
neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors.fit(X_model)
distances, indices = neighbors.kneighbors(X_model)
k_distances = np.sort(distances[:, -1])

ax = pd.Series(k_distances).plot()
ax.set_title(f"k-distance plot, k = {min_samples}")
ax.set_xlabel("Points sorted by distance")
ax.set_ylabel(f"Distance to {min_samples}-th nearest neighbor")
plt.show()


Zoom into the upper tail of the plot, where the elbow is usually easier to see.


In [ ]:
tail_start = int(0.85 * len(k_distances))
ax = pd.Series(k_distances[tail_start:]).reset_index(drop=True).plot()
ax.set_title("Upper tail of the k-distance plot")
ax.set_xlabel("Tail points sorted by distance")
ax.set_ylabel(f"Distance to {min_samples}-th nearest neighbor")
plt.show()


In [ ]:
print("Useful percentiles for eps exploration:")
for q in [85, 90, 92, 95, 97, 99]:
    print(q, np.percentile(k_distances, q).round(4))


Use the plot and percentiles to set a starting value. If the model returns almost no noise points, **reduce** `eps`. If it returns almost every point as noise, **increase** `eps`.


In [ ]:
chosen_eps = 3.5  # Replace this value after reviewing your k-distance plot.
print("Chosen eps:", chosen_eps)
print("Chosen min_samples:", min_samples)


### Grading evidence — 20 pts

- **7 pts**: The k-distance plot is generated.
- **5 pts**: The upper-tail plot or percentile table is included.
- **5 pts**: A specific `eps` value is selected.
- **3 pts**: The selected `eps` and `min_samples` are stated before fitting DBSCAN.


## 5. Fitting DBSCAN and visualizing anomalies — 20 pts

Fit DBSCAN using the PCA modeling matrix. DBSCAN labels noise points as **-1**. Treat those noise points as possible anomalies.


In [ ]:
from sklearn.cluster import DBSCAN

model = DBSCAN(
    eps=chosen_eps,
    min_samples=min_samples,
    metric="euclidean",
    n_jobs=-1
)
labels = model.fit_predict(X_model)

work_df["dbscan_label"] = labels
work_df["is_noise"] = (labels == -1).astype(int)
work_df["k_distance"] = distances[:, -1]

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int(work_df["is_noise"].sum())
noise_rate = work_df["is_noise"].mean()

print("Number of clusters:", n_clusters)
print("Noise points:", n_noise)
print("Noise rate:", round(noise_rate, 4))


Visualize the model output on the PCA projection.


In [ ]:
pca_df["dbscan_label"] = work_df["dbscan_label"].values
pca_df["is_noise"] = work_df["is_noise"].values

ax = pca_df.plot(kind="scatter", x="PC1", y="PC2", c="is_noise", s=8, alpha=0.6)
ax.set_title("DBSCAN noise points on PCA projection")
plt.show()


Inspect the observations that look most isolated according to the k-distance value.


In [ ]:
columns_to_view = ["Time", "Amount", "Class", "dbscan_label", "is_noise", "k_distance"]
top_isolated = (
    work_df[columns_to_view]
    .sort_values("k_distance", ascending=False)
    .head(15)
)
display(top_isolated)


Run a small **sensitivity analysis**. This does not replace the k-distance plot; it helps you observe how `eps` changes the number of noise points and the detection metrics.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

eps_values = np.arange(2.0, 6.1, 0.5)
results = []
for eps in eps_values:
    temp_model = DBSCAN(eps=float(eps), min_samples=min_samples, metric="euclidean", n_jobs=-1)
    temp_labels = temp_model.fit_predict(X_model)
    temp_pred = (temp_labels == -1).astype(int)
    temp_clusters = len(set(temp_labels)) - (1 if -1 in temp_labels else 0)
    results.append({
        "eps": round(float(eps), 2),
        "clusters": temp_clusters,
        "noise_points": int(temp_pred.sum()),
        "noise_rate": round(float(temp_pred.mean()), 4),
        "precision": round(precision_score(y, temp_pred, zero_division=0), 4),
        "recall": round(recall_score(y, temp_pred, zero_division=0), 4),
        "f1": round(f1_score(y, temp_pred, zero_division=0), 4)
    })

results_df = pd.DataFrame(results)
display(results_df)

ax = results_df.plot(x="eps", y=["precision", "recall", "f1"], marker="o")
ax.set_title("Sensitivity analysis for eps")
ax.set_ylabel("Metric value")
plt.show()


### Grading evidence — 20 pts

- **5 pts**: DBSCAN is fitted correctly using scaled features.
- **4 pts**: Noise points are converted into an anomaly flag.
- **4 pts**: Number of clusters, number of noise points, and noise rate are reported.
- **4 pts**: PCA visualization of noise points is included.
- **3 pts**: Sensitivity analysis over several `eps` values is included.


## 6. Evaluation against the fraud label — 15 pts

Use `Class` **only now**. The model has already produced an anomaly flag without seeing the label. Evaluate how often DBSCAN noise points match actual fraud transactions.

> **Note:** `Class` was **not** used during DBSCAN fitting — only for evaluation.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = work_df["is_noise"].astype(int)
cm = confusion_matrix(y, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["Actual legitimate", "Actual fraud"],
    columns=["Predicted clustered", "Predicted noise"]
)
display(cm_df)

print(classification_report(
    y,
    y_pred,
    target_names=["Legitimate", "Fraud"],
    zero_division=0
))


Compute a compact fraud-detection summary.


In [ ]:
actual_frauds = int((y == 1).sum())
detected_frauds = int(((y == 1) & (y_pred == 1)).sum())
false_alerts = int(((y == 0) & (y_pred == 1)).sum())
missed_frauds = int(((y == 1) & (y_pred == 0)).sum())

summary = pd.DataFrame({
    "metric": [
        "Actual frauds",
        "Detected frauds as noise",
        "Missed frauds",
        "False alerts",
        "Fraud recall",
        "Alert precision"
    ],
    "value": [
        actual_frauds,
        detected_frauds,
        missed_frauds,
        false_alerts,
        round(detected_frauds / actual_frauds, 4) if actual_frauds else np.nan,
        round(detected_frauds / (detected_frauds + false_alerts), 4) if (detected_frauds + false_alerts) else np.nan
    ]
})
display(summary)


### Technical interpretation

Complete the following after running the evaluation cells:

1. The selected `eps` and `min_samples` values.
2. The number and percentage of transactions marked as noise.
3. Precision, recall, and F1-score for the fraud class.
4. One sentence explaining why a noise point is not necessarily a fraud case.


*(Write your interpretation here.)*

1. **Parameters:** `eps = ...`, `min_samples = ...`
2. **Noise:** ... transactions (...%)
3. **Fraud class metrics:** precision = ..., recall = ..., F1 = ...
4. **Noise ≠ fraud:** ...


### Grading evidence — 15 pts

- **4 pts**: Confusion matrix is displayed with clear row and column labels.
- **4 pts**: Precision, recall, and F1-score are reported.
- **3 pts**: The compact fraud-detection summary is included.
- **3 pts**: The interpretation text cell includes all requested items.
- **1 pt**: The notebook clearly states that `Class` was not used for DBSCAN fitting.


## 7. Notebook organization and reproducibility — 5 pts

Before submitting, **restart the notebook kernel** and run all cells from the beginning. The notebook must run without manual changes after the dataset file is available.

### Grading evidence — 5 pts

- **2 pts**: Code cells are executed in order and outputs are visible.
- **1 pt**: Variable names are clear and consistent.
- **1 pt**: The notebook includes short markdown explanations before each major step.
- **1 pt**: The final notebook is clean, readable, and does not include unrelated experiments.

---

## Score summary

| Section | Points |
|---|---|
| Setup and dataset access | 10 |
| Data loading and first inspection | 15 |
| Feature preparation | 15 |
| Choosing DBSCAN parameters | 20 |
| Fitting DBSCAN and visualizing anomalies | 20 |
| Evaluation against the fraud label | 15 |
| Notebook organization and reproducibility | 5 |
| **Total** | **100** |

## References

- Ester, M., Kriegel, H.-P., Sander, J., and Xu, X. (1996). *A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise.* Proceedings of KDD-96, 226–231.
- Kaggle/ULB. Credit Card Fraud Detection dataset. https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
- Scikit-learn. DBSCAN documentation. https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html
- Porwal, U., and Mukund, S. (2018). *Credit Card Fraud Detection in e-Commerce: An Outlier Detection Approach.* https://arxiv.org/abs/1811.02196
- Niu, X., Wang, L., and Yang, X. (2019). *A Comparison Study of Credit Card Fraud Detection: Supervised versus Unsupervised.* https://arxiv.org/abs/1904.10604
